# Embeddings, Clustering, and Theme Extraction

Turning thousands of transcript responses into named, countable themes.

This notebook is an original tutorial extending the course with current
(2026) state-of-the-art practice, interview-focused for insight-synthesis
products (AI interview platforms, feedback analysis).

## Learning Objectives

- Explain what embeddings are and how cosine similarity search works mechanically.
- Name the current embedding models and the Matryoshka truncation trick.
- Describe the embedding-cluster pipeline (UMAP + HDBSCAN, BERTopic-style) and its tradeoffs.
- Describe the LLM-taxonomy approach (Clio-style) and why the hybrid is current best practice.
- Choose a vector database sensibly (pgvector is usually enough) and explain HNSW vs IVF at a concept level.

## Mental Model

Theme extraction is a two-question problem:

1. **What themes exist?** (discovery — unsupervised, catches unknown-unknowns)
2. **How many people said each?** (measurement — needs every response
   classified against a *fixed* taxonomy, so counts are exact and comparable)

Embedding-clustering is good at question 1 and bad at question 2 (clusters
are fuzzy, 10-30% lands in noise). LLM classification against a taxonomy is
good at question 2 but only as good as the taxonomy. The 2026 answer is the
hybrid: **cluster to propose, LLM to name and merge, cheap LLM to classify
everything against the frozen taxonomy.**

## Important Concepts

- **Embedding**: text → dense vector such that semantic similarity ≈ geometric closeness; normalize to unit length so cosine similarity = dot product.
- **Matryoshka (MRL)**: models trained to front-load meaning into early dimensions; truncate 3072→512 dims + renormalize with minimal recall loss; combine with quantization for ~78%+ storage cuts.
- **ANN search**: HNSW (layered graph, greedy descent, high recall, memory-hungry — the default) vs IVF (k-means cells, probe the nearest few, cheaper but needs retraining).
- **UMAP**: reduce ~1500 dims to 5-15 before density clustering (distance concentration breaks HDBSCAN in high dims).
- **HDBSCAN**: density-based, no preset k, marks outliers as noise — a feature: not everything is a theme.
- **Clio-style pipeline** (Anthropic): cheap model summarizes each conversation into a facet → embed + cluster facets → strong model names clusters → merge into hierarchy → classify everything into the taxonomy.

## Need-To-Know Coverage Checklist

- [x] Cosine similarity mechanics; exact vs ANN search.
- [x] Current models: OpenAI text-embedding-3-large/small, Voyage, Cohere embed-v4, Gemini embedding, bge-m3.
- [x] Matryoshka truncation + quantization.
- [x] Vector DB landscape: pgvector, Qdrant, Turbopuffer, Pinecone, Weaviate — and when pgvector is simply enough.
- [x] BERTopic-style pipeline: embed → UMAP → HDBSCAN → LLM labels.
- [x] LLM-taxonomy / Clio-style pipeline and cost structure.
- [x] Tradeoffs table: embedding-cluster vs LLM-taxonomy; the hybrid.

## Deep Study Notes

### Embedding models (mid-2026)

| Model | Dims | Price /1M tok | Notes |
|---|---|---|---|
| OpenAI text-embedding-3-large | 3072 (MRL) | $0.13 | safe default |
| OpenAI text-embedding-3-small | 1536 | $0.02 | cheapest big-provider |
| Voyage voyage-4 | 1024-2048 | $0.06-0.12 | best when retrieval quality is the bottleneck |
| Cohere embed-v4 | 1536 (MRL) | ~$0.01-0.12 | multimodal (text+images) |
| Gemini embedding | 3072 (MRL) | cheap/free tier | tops public MTEB |
| bge-m3 / gte / nomic (OSS) | 384-1024 | self-host | multilingual self-host default |

### Similarity search, mechanically

Each text becomes a vector; normalize to unit length; similarity is the dot
product (= cosine of the angle). Exact nearest-neighbor is O(N·d) per query;
at scale use ANN:
- **HNSW**: multi-layer navigable-small-world graph; greedy search from a
  sparse top layer down; roughly logarithmic queries, high recall, high memory.
- **IVF**: k-means partitions; search only the nProbe nearest cells; cheaper,
  lower recall, needs retraining as data drifts.

### Vector DBs — the honest sizing answer

5,000 interviews × 50 utterances = 250k vectors. **pgvector handles this
trivially** (HNSW since 0.5.0; 8-25ms p95) and lets you *join* vectors with
relational data — interview → respondent → segment filters, which is exactly
what a research dashboard needs. Qdrant for best self-hosted filtering/price,
Turbopuffer for object-storage-scale corpora, Pinecone for easiest managed,
Weaviate when you want hybrid search and built-in vectorizer modules in one
managed package (middle of the pack on price/performance).
Reaching for a dedicated vector DB at 250k vectors is a red-flag answer.

### Pipeline A: embedding + clustering (BERTopic-style)

chunk into utterances → embed → **UMAP** to 5-15 dims → **HDBSCAN** →
label clusters with an LLM given representative docs + c-TF-IDF keywords.
Variants: k-means when you need fixed k and full coverage; agglomerative for
a theme hierarchy.

Weaknesses: clusters follow *semantic similarity*, not analyst-relevant
distinctions ("pricing complaint" and "pricing praise" co-cluster); labels can
be mushy; HDBSCAN dumps 10-30% into noise; parameter-sensitive.

### Pipeline B: LLM taxonomy (Clio-style)

1. Cheap model (Haiku-class) extracts a one-line facet summary per response.
2. Embed + cluster the facet summaries (k-means is fine here).
3. Strong model names each cluster; recursively merge bottom-up into a hierarchy.
4. **Classify every response** into the frozen taxonomy with a cheap model
   (structured output) → exact per-theme counts + evidence.

Cost shape: one expensive taxonomy pass + thousands of cheap classify calls.
Simplest variant: sample N transcripts → frontier model proposes taxonomy →
cheap model classifies everything.

Weaknesses: taxonomy quality depends on the sample; can miss rare emergent
themes; re-running yields different taxonomies — **version and freeze them**.

### The hybrid (current best practice)

Cluster embeddings to *propose* candidate themes (unknown-unknowns) → LLM
*refines, merges, and names* them into a versioned taxonomy → cheap LLM
*classifies* every unit against it (counts + evidence spans) → periodically
re-cluster the "uncategorized" bucket to catch drift.

## Common Failure Modes

- Clustering raw high-dim embeddings with HDBSCAN → everything is noise (distance concentration).
- Treating cluster size as respondent count → clusters are utterance-level and fuzzy; counts need classification.
- No `uncategorized` handling → new themes invisible until someone reruns everything.
- Unfrozen taxonomy → week-over-week dashboards not comparable.
- Sentiment-blind clustering → praise and complaints merged into one "pricing" theme.
- Picking Pinecone at 250k vectors → cost and operational complexity for nothing.

## Implementation Notes

- Embed at full dimension; store truncated+quantized; rerank top-k with full vectors if needed.
- Store `taxonomy_version` on every classification row.
- Keep evidence: every theme assignment carries the utterance id and quote span.
- Batch embedding calls; they are the cheap part — classification tokens dominate cost.

In [ ]:
"""Tiny end-to-end theme pipeline: embed -> cluster -> name -> classify.

Uses toy 2-D "embeddings" so it runs anywhere; the structure is exactly the
production pipeline (swap in real embeddings + HDBSCAN + LLM calls).
"""
import math
from collections import defaultdict

# Toy corpus: utterances with hand-set 2-D embeddings (real: 1024-3072 dims)
UTTERANCES = [
    ("u1", "The signup flow kept erroring out", (0.9, 0.1)),
    ("u2", "I could not finish creating my account", (0.85, 0.15)),
    ("u3", "Onboarding took me three tries", (0.8, 0.2)),
    ("u4", "The price is way too high for what you get", (0.1, 0.9)),
    ("u5", "I don't understand what the tiers include", (0.15, 0.85)),
    ("u6", "Support never answered my email", (0.5, 0.5)),
]


def cosine(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = math.sqrt(sum(x * x for x in a))
    nb = math.sqrt(sum(x * x for x in b))
    return dot / (na * nb)


# --- 1. Cluster (toy: greedy threshold; production: UMAP + HDBSCAN) ------
def cluster(items, threshold=0.95):
    clusters = []
    for uid, text, vec in items:
        for c in clusters:
            if cosine(vec, c["centroid"]) >= threshold:
                c["members"].append((uid, text))
                break
        else:
            clusters.append({"centroid": vec, "members": [(uid, text)]})
    return clusters


clusters = cluster(UTTERANCES)

# --- 2. Name clusters (production: LLM given representative docs) --------
NAMES = {0: "onboarding friction", 1: "pricing confusion", 2: "support responsiveness"}
# .get() fallback: a new cluster (e.g. from Practice Q1) lands in an
# "unnamed" bucket instead of crashing — mirroring the real 'uncategorized'
# bucket a production taxonomy needs.
taxonomy = {NAMES.get(i, f"unnamed-cluster-{i}"): [uid for uid, _ in c["members"]]
            for i, c in enumerate(clusters)}

# --- 3. Classify everything against the frozen taxonomy -> exact counts --
counts = {theme: len(uids) for theme, uids in taxonomy.items()}
print("taxonomy v1 (frozen):")
for theme, uids in taxonomy.items():
    print(f"  {theme}: n={len(uids)}, evidence={uids}")

# Small clusters are 'emerging signals', not established themes
for theme, n in counts.items():
    tier = "established" if n >= 3 else "emerging signal (low n)"
    print(f"  -> {theme}: {tier}")


## Practice

1. Add a new utterance about "billing surprises". Does it join pricing or
   form a new cluster at threshold 0.95? What does that tell you about
   threshold sensitivity vs HDBSCAN's density approach?
2. Split the pricing cluster by sentiment: what extra signal (beyond the
   embedding) do you need, and where in the pipeline would you add it?
3. Estimate the cost of the Clio-style pipeline for 5,000 interviews:
   one facet extraction per interview (cheap model), one taxonomy pass over
   500 sampled facets (frontier), one classification per utterance (cheap).
4. Defend pgvector over Pinecone for this product in two sentences, using the
   join-with-relational-data argument.

## Design Checklist

- [ ] UMAP (or similar) before density clustering; never HDBSCAN raw high-dim vectors.
- [ ] Taxonomy is versioned and frozen per reporting period.
- [ ] Every theme assignment carries evidence (utterance id + quote span).
- [ ] Counts come from classification, never from cluster sizes.
- [ ] `uncategorized` bucket exists and is periodically re-clustered.
- [ ] Vector store sized honestly (pgvector until ~5-10M vectors).
- [ ] Embeddings truncated/quantized for storage; full-dim rerank if needed.